In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import math
import statistics

import warnings
warnings.filterwarnings('ignore')

#sys.path.append('../common')
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold,StratifiedShuffleSplit
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import make_scorer,recall_score


from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# #-----
#from xgboost.sklearn import XGBClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split


# import pprint

# from scipy import stats
import scipy.io as spio
import numpy as np
import pandas as pd
# import pprint
# import scipy.io as spio
# from tqdm import tqdm
# from timeit import default_timer as timer
import pickle



# pp = pprint.PrettyPrinter(indent=4, depth=3)

# plt.style.use('ggplot')
import category_encoders as ce
import lightgbm
import logging
import numpy as np
import os
import pandas as pd
#import pytest
import sklearn
import xgboost

import ads

from ads.hpo.stopping_criterion import *
from ads.hpo.distributions import *
from ads.hpo.search_cv import ADSTuner

from lightgbm import LGBMClassifier
from sklearn import preprocessing
from sklearn.compose import ColumnTransformer
#from sklearn.datasets import load_iris, load_boston
from sklearn.decomposition import PCA
from sklearn.ensemble import AdaBoostRegressor, AdaBoostClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.metrics import make_scorer, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from xgboost import XGBClassifier
from ads.model import GenericModel
from ads.model import DataScienceModel
import automlx
from automlx import init


In [ ]:
# outFile = open('demo.txt', 'w')
# pd.set_option('display.max_rows', 999)
# pd.set_option('display.max_columns', 999)
# pd.set_option('display.width', 999)
# np.set_printoptions(threshold=sys.maxsize)
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

<H3> Data </H3>

In [ ]:
blinks = pd.read_csv(os.path.join(os.getcwd(),'data/DLC_blink_features.csv'), sep=',', header=0)

blinks_df = pd.DataFrame(blinks)


In [ ]:
import pandas as pd
from sklearn import preprocessing

import pandas as pd

# # Load the two CSV files, each with 1 row
# mean_df=pd.read_csv(os.path.join(os.getcwd(),'data/all_mean_outliers2.csv'), sep=',', header=0)
label_df=blinks_df['scores']
# mean_df = blinks_df.drop(['Unnamed: 0','Outer Angle Min','label'],axis=1);
mean_df=blinks_df
mean_df.drop(columns=['filename','scores'], inplace=True)
# cols_to_keep =[
       #'EAR Max', 'Inner Angle Min', 'Inner Angle Max',
       #'Outer Angle Max', 'Velocity Max', 'Eye Area Max']

cols_to_keep =[
        'min_p4', 'max_euclid_dist', 'max_ear', 'max_eye_area', 'max_p1', 'max_p4']

mean_df.drop(columns=blinks_df.columns.difference(cols_to_keep), inplace=True)

cols= mean_df.columns

x = mean_df.values #returns a numpy array
min_max_scaler = preprocessing.MinMaxScaler()
x_scaled = min_max_scaler.fit_transform(x)
mean_df = pd.DataFrame(x_scaled)
mean_df.columns=cols

mean_df

X=mean_df

y=label_df

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

<H1>Auto ML </H1>

In [ ]:
# init(engine='local',check_deprecation_warnings = False)
init(engine='ray')

In [ ]:
est1 = automlx.Pipeline(task='classification',model_list=[ 'SVC', 'KNeighborsClassifier','XGBClassifier', 'GaussianNB','LogisticRegression','LGBMClassifier', 'CatBoostClassifier'],score_metric=['accuracy','precision','recall','roc_auc'])
# model_list=[ 'SVC', 'KNeighborsClassifier','XGBClassifier', 'GaussianNB', 'LogisticRegression', 'LGBMClassifier', 'CatBoostClassifier'],

est1.fit(X_train, y_train, cv=4)


In [ ]:
est1.print_summary()

In [ ]:
explainer = automlx.MLExplainer(est1,
                               X,
                               y,
                               task="classification")


In [ ]:
result_explain_model_default = explainer.explain_model(
    n_iter=5,                            # Can also be 'auto' to pick a good value for the explainer and task

    scoring_metric='balanced_accuracy',  # Global feature importance measures how much each feature improved the
                                         # model's score. Users can chose the scoring metric used here.
)
dd=result_explain_model_default.to_dataframe()
dd


In [ ]:
X.ads.feature_stat()

In [ ]:
X.ads.feature_stat().pivot_table(index='Column', columns='Metric', values = 'Value')

In [ ]:
# model = KNeighbors() ##Initialize the model

# tuner = ADSTuner(model, cv=3) ## cv is cross validation splits
# tuner.search_space() ##This is the default search space
# tuner.tune(X_OH, y_OH, exit_criterion=[NTrials(10)])

In [ ]:
df_result = est1.completed_trials_detailed_
df_result
df_result.to_csv('results_ml_v4.csv')
df_result.sort_values('Score (accuracy)',ascending=False)

In [ ]:
df_result

In [ ]:
from sklearn.metrics import accuracy_score, recall_score, roc_auc_score, roc_curve, precision_score, RocCurveDisplay

#fpr, tpr, thresholds = metrics.roc_curve(y, scores, pos_label=2)
#sklearn.metrics.roc_curve(y_true, y_score, pos_label=None, sample_weight=None, drop_intermediate=True)

y_pred = est1.predict(X_test)
y_pred_proba = est1.predict_proba(X_test)[:, 1] # Probability of the positive class

ranked_models = est1.ranked_models_
selected_model = est1.selected_model_

RocCurveDisplay.from_predictions(y_test, y_pred_proba)

#Add diagonal line (chance line)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chance')
plt.rcParams['axes.linewidth'] = 2.0
plt.xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0], ['0%', '20%', '40%', '60%', '80%', '100%'],fontsize=14)
plt.yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0], ['0%', '20%', '40%', '60%', '80%', '100%'],fontsize=14)
plt.ylabel('True Positive Rate (%)',fontsize=16)
plt.xlabel('False Positive Rate (%)',fontsize=16)

plt.legend(fontsize=14)

plt.savefig(os.path.join("est1_roc_curve.png"), dpi=300, bbox_inches="tight")
plt.savefig(os.path.join("est1_roc_curve.pdf"), dpi=300, bbox_inches="tight")
plt.savefig(os.path.join("est1_roc_curve.tif"), dpi=300, bbox_inches="tight")
plt.show()

score_accuracy = accuracy_score(y_test, y_pred)
score_roc_auc = roc_auc_score(y_test, y_pred_proba)
score_recall = recall_score(y_test, y_pred)
score_precision = precision_score(y_test,y_pred)

# y_proba = est1.predict_proba(X_test)
# score_roc_auc = roc_auc_score(y_test, y_proba[:, 1])
# score_acc = accuracy_score(y_test, y_proba[:, 1])

print(f'Score on test data: Accuracy : {score_accuracy}')
print(f'Score on test data ROC AUC : {score_roc_auc}')
print(f'Score on test data Recall : {score_recall}')
print(f'Score on test data Precision : {score_precision}')

# Save figure and calculations
#1. Organize metrics into a dictionary
metrics_dict = { "Metric": ["Accuracy", "ROC AUC", "Recall", "Precision"], "Score": [score_accuracy, score_roc_auc, score_recall, score_precision], }
#2. Convert to a structured table DataFrame
df = pd.DataFrame(metrics_dict)
#3. Save to a CSV file (overwrites the file each time you run it)
df.to_csv("model_evaluation_scores.csv", index=False)



In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Set smaller, compact font sizes for publication
plt.rcParams.update({
    'font.size': 9,          # Numbers inside the matrix
    'axes.labelsize': 9,     # 'True label' and 'Predicted label'
    'axes.titlesize': 9,     # Title size
    'xtick.labelsize': 8,    # Class names on x-axis
    'ytick.labelsize': 8     # Class names on y-axis
})

# Generate predictions and confusion matrix
y_pred = est1.predict(X_test)
cm = confusion_matrix(y_test, y_pred, labels=est1.classes_)

# Create the display object
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=est1.classes_)

# Smaller figure canvas with high resolution for crisp text
fig, ax = plt.subplots(figsize=(2.5, 2.5), dpi=300)

# Plot the matrix
disp.plot(cmap='gray', ax=ax, values_format='d')

# Prevent text clipping
plt.tight_layout()

# Save in both formats with tight bounding boxes
plt.savefig('confusion_matrix.png', bbox_inches='tight', dpi=300)

plt.show()


In [ ]:
result_explain_model_default.show_in_notebook()

In [ ]:
trials = est1.completed_trials_summary_[est1.completed_trials_summary_["Step"].str.contains('Model Selection')]
name_of_score_column = f"Score ({est1._inferred_score_metric[0].name})"
trials.replace([np.inf, -np.inf], np.nan, inplace=True)
trials.dropna(subset=[name_of_score_column], inplace = True)
scores = trials[name_of_score_column].tolist()
models = trials['Algorithm'].tolist()
colors = []
patterns = []

y_margin = 0.10 * (max(scores) - min(scores))
s = pd.Series(scores, index=models).sort_values(ascending=False)
s = s.dropna()
for f in s.keys():
    if f.strip()  ==  est1.selected_model_.strip():
        colors.append('lightblue')
        # colors.append('orange')
        patterns.append('')
    elif s[f] >= s.mean():
        colors.append('lightgrey')
        # colors.append('teal')
        patterns.append('\\\\')
    else:
        colors.append('grey')
        # colors.append('turquoise')
        patterns.append('/')

fig, ax = plt.subplots(1)
s.plot.barh(ax=ax, color=colors, edgecolor='black', hatch=patterns)
plt.tick_params(axis='both', which='major', labelsize=14)
plt.rcParams['axes.linewidth'] = 0.75
ax.set_title("Algorithm Selection Trials")
ax.set_xlim(min(scores) - y_margin, max(scores) + y_margin)
# ax.set_ylabel(est1._inferred_score_metric[0].name,fontsize=16)
ax.set_xlabel("Classification Accuracy (%)",fontsize=16)
ax.set_xticks(np.arange(0,1.1,0.1), labels=range(0,110,10),fontsize=14)
ax.axvline(x=s.mean(), color='black',linestyle='--',linewidth=1.0)

plt.savefig('classifier_accuracies.png', bbox_inches='tight', dpi=300)
plt.savefig('classifier_accuracies.pdf', bbox_inches='tight', dpi=300)
plt.savefig('classifier_accuracies.tiff', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
est1.print_summary()